<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - Building Agents with the OpenAI Agents SDK and MLflow

## Overview
Now that you know how to create and govern agent tools, this lecture introduces the framework you will use to build agents: the **OpenAI Agents SDK** running on Databricks. We cover orchestration patterns, MCP integration, the `@function_tool` decorator, multi-agent concepts, and how MLflow provides observability across single-agent and multi-agent flows.

## Learning Objectives
By the end of this lecture, you will be able to:
1. Identify the orchestration patterns supported by the OpenAI Agents SDK (handoffs, agents-as-tools, sequential, parallel, feedback loops)
2. Explain how `McpServer` connects agents to Unity Catalog functions via MCP
3. Describe how `@function_tool` turns a Python function into an agent tool
4. Understand multi-agent concepts: supervisor agents, handoffs, and worker specialization
5. Recognize other supported frameworks (LangChain, LangGraph, DSPy) and when to use them
6. Explain how MLflow tracing captures agent execution as hierarchical spans

## A. OpenAI Agent Orchestration Patterns

[The OpenAI Agents SDK](https://openai.github.io/openai-agents-python/multi_agent/) supports multiple patterns for coordinating agents. These fall into two categories: **LLM-driven** (the model decides what happens next) and **code-driven** (your code controls the flow deterministically).

**Click on each pattern** to see how it works.

<style>
.orch-outer { max-width: 1100px; margin: 0 auto; font-family: sans-serif; font-size: 12pt; }
.orch-label { font-size: 12pt; font-weight: 600; color: #999; letter-spacing: 0.5px; text-transform: uppercase; margin-bottom: 8px; }
.orch-row { display: flex; flex-wrap: wrap; gap: 10px; margin-bottom: 14px; }
.orch-node {
  flex: 1; min-width: 140px; background: #F9F7F4; border-top: 6px solid transparent;
  border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent;
  border-radius: 8px; padding: 14px 12px; text-align: center; cursor: pointer; user-select: none;
  transition: transform 0.12s, background 0.15s;
}
.orch-node:hover { transform: translateY(-2px); }
.orch-node.active { background: #fff; border-left-color: var(--oc); border-right-color: var(--oc); border-bottom-color: var(--oc); }
.orch-node-title { display: block; font-size: 12pt; font-weight: 700; color: #0b2026; line-height: 1.3; pointer-events: none; }
.orch-node-sub { display: block; font-size: 12pt; color: #666; margin-top: 4px; pointer-events: none; }
.orch-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease; }
.orch-detail-wrap.open { max-height: 1600px; opacity: 1; margin-top: 12px; }
.orch-detail-card { background: #F9F7F4; border-radius: 10px; padding: 20px; border-top: 6px solid #ccc; font-size: 12pt; line-height: 1.7; color: #333; }
.orch-detail-card strong { color: #0b2026; }
.orch-detail-card code { background: #e8e8e8; padding: 2px 6px; border-radius: 4px; font-size: 12pt; }
.orch-flow { display: flex; align-items: center; justify-content: center; gap: 0; margin: 14px 0; flex-wrap: wrap; }
.orch-flow-box { background: #1e1e2e; color: #fff; border-radius: 6px; padding: 8px 14px; font-size: 12pt; font-weight: 600; text-align: center; min-width: 90px; }
.orch-flow-arrow { color: #999; font-size: 20px; padding: 0 6px; }
</style>

<div class="orch-outer">

<div class="orch-label">LLM-Driven Orchestration</div>
<div class="orch-row">
  <div class="orch-node" data-orch="handoffs" onclick="orchSelect('handoffs')" style="--oc:#FF5F46; border-top-color:#FF5F46;">
    <span class="orch-node-title">Handoffs</span>
    <span class="orch-node-sub">Specialist takes over</span>
  </div>
  <div class="orch-node" data-orch="tools" onclick="orchSelect('tools')" style="--oc:#4299E0; border-top-color:#4299E0;">
    <span class="orch-node-title">Agents as Tools</span>
    <span class="orch-node-sub">Manager calls specialists</span>
  </div>
</div>

<div class="orch-label">Code-Driven Orchestration</div>
<div class="orch-row">
  <div class="orch-node" data-orch="sequential" onclick="orchSelect('sequential')" style="--oc:#00A972; border-top-color:#00A972;">
    <span class="orch-node-title">Sequential Chaining</span>
    <span class="orch-node-sub">Pipeline of agents</span>
  </div>
  <div class="orch-node" data-orch="parallel" onclick="orchSelect('parallel')" style="--oc:#FFAB00; border-top-color:#FFAB00;">
    <span class="orch-node-title">Parallel Execution</span>
    <span class="orch-node-sub">Concurrent agents</span>
  </div>
  <div class="orch-node" data-orch="feedback" onclick="orchSelect('feedback')" style="--oc:#1B5162; border-top-color:#1B5162;">
    <span class="orch-node-title">Feedback Loops</span>
    <span class="orch-node-sub">Iterate until quality</span>
  </div>
</div>

<div class="orch-detail-wrap" id="orch-detail-wrap">
  <div class="orch-detail-card" id="orch-detail-card"></div>
</div>

</div>

<script>
var ORCH = {
  handoffs: {
    color: '#FF5F46',
    title: 'Handoffs',
    html: '<strong>Pattern:</strong> A triage/supervisor agent routes the conversation to a specialist, which then <strong>becomes the active agent</strong> and responds directly to the user.<br/><br/><strong>Key behavior:</strong> Conversation ownership transfers. The specialist takes over; the supervisor does not post-process.<br/><br/><div class="orch-flow"><div class="orch-flow-box">User</div><div class="orch-flow-arrow">&#x2192;</div><div class="orch-flow-box" style="background:#FF5F46;">Supervisor</div><div class="orch-flow-arrow">&#x2192;</div><div class="orch-flow-box" style="background:#2574B5;">Specialist A</div><div class="orch-flow-arrow">&#x2192;</div><div class="orch-flow-box">User</div></div><strong>When to use:</strong> The specialist should respond directly, prompts need to stay focused, or you want to swap instructions and tool sets per route.<br/><br/><strong>SDK pattern:</strong> <code>Agent(handoffs=[agent_a, agent_b])</code><br/><br/>'
  },
  tools: {
    color: '#4299E0',
    title: 'Agents as Tools',
    html: '<strong>Pattern:</strong> A manager agent maintains conversation control and invokes specialist agents as tools via <code>Agent.as_tool()</code>. The manager <strong>synthesizes the results</strong> from multiple specialists into a final answer.<br/><br/><strong>Key behavior:</strong> The manager always owns the conversation. Specialists are called and their outputs are combined.<br/><br/><div class="orch-flow"><div class="orch-flow-box">User</div><div class="orch-flow-arrow">&#x2192;</div><div class="orch-flow-box" style="background:#4299E0;">Manager</div><div class="orch-flow-arrow">&#x2194;</div><div style="display:flex;flex-direction:column;gap:4px;"><div class="orch-flow-box" style="background:#00A972;">Tool Agent A</div><div class="orch-flow-box" style="background:#FFAB00;">Tool Agent B</div></div><div class="orch-flow-arrow">&#x2192;</div><div class="orch-flow-box">User</div></div><strong>When to use:</strong> You want one agent to own the final answer, combine outputs from multiple specialists, or maintain a single conversational thread.<br/><br/><strong>SDK pattern:</strong> <code>Agent(tools=[specialist.as_tool()])</code>'
  },
  sequential: {
    color: '#00A972',
    title: 'Sequential Chaining',
    html: '<strong>Pattern:</strong> Your code pipes the output of one agent as the input to the next. Each agent handles one step in a multi-step pipeline.<br/><br/><strong>Key behavior:</strong> Deterministic order. Each agent runs exactly once in sequence.<br/><br/><div class="orch-flow"><div class="orch-flow-box" style="background:#00A972;">Research</div><div class="orch-flow-arrow">&#x2192;</div><div class="orch-flow-box" style="background:#00A972;">Outline</div><div class="orch-flow-arrow">&#x2192;</div><div class="orch-flow-box" style="background:#00A972;">Draft</div><div class="orch-flow-arrow">&#x2192;</div><div class="orch-flow-box" style="background:#00A972;">Review</div></div><strong>When to use:</strong> Tasks decompose naturally into discrete steps, each requiring different instructions or tools. You want full control over the order of execution.<br/><br/><strong>Code pattern:</strong> <code>result1 = await Runner.run(agent1, input); result2 = await Runner.run(agent2, result1.to_input_list())</code><br/><br/>Use <code>result1.to_input_list()</code> rather than <code>result1.final_output</code> to carry the full conversation history forward, not just the final text string.'
  },
  parallel: {
    color: '#FFAB00',
    title: 'Parallel Execution',
    html: '<strong>Pattern:</strong> Run multiple independent agents simultaneously using <code>asyncio.gather</code>. Results are collected and combined after all agents finish.<br/><br/><strong>Key behavior:</strong> Concurrent execution. Faster than sequential when tasks are independent.<br/><br/><div class="orch-flow"><div class="orch-flow-box">Input</div><div class="orch-flow-arrow">&#x2192;</div><div style="display:flex;flex-direction:column;gap:4px;"><div class="orch-flow-box" style="background:#FFAB00;">Agent A</div><div class="orch-flow-box" style="background:#FFAB00;">Agent B</div><div class="orch-flow-box" style="background:#FFAB00;">Agent C</div></div><div class="orch-flow-arrow">&#x2192;</div><div class="orch-flow-box">Combine</div></div><strong>When to use:</strong> Multiple independent analyses of the same input, or different perspectives that can run concurrently without depending on each other.<br/><br/><strong>Code pattern:</strong> <code>results = await asyncio.gather(Runner.run(a1, q), Runner.run(a2, q), Runner.run(a3, q))</code>'
  },
  feedback: {
    color: '#1B5162',
    title: 'Feedback Loops',
    html: '<strong>Pattern:</strong> An agent runs in a loop, paired with an evaluator agent that provides feedback. The evaluator\'s feedback is passed back to the generator each iteration. The loop continues until the evaluator approves the output or a max iteration count is reached.<br/><br/><strong>Key behavior:</strong> Iterative refinement. Quality improves with each pass because the generator receives explicit feedback, not just a retry signal.<br/><br/><div class="orch-flow"><div class="orch-flow-box" style="background:#1B5162;">Generator</div><div class="orch-flow-arrow">&#x2192;</div><div class="orch-flow-box" style="background:#1B5162;">Evaluator</div><div class="orch-flow-arrow">&#x21BA;</div><div class="orch-flow-box" style="font-size:11px;">Pass?</div></div><strong>When to use:</strong> Tasks where quality can be measured programmatically or by an LLM judge, and iterative improvement is worth the extra cost (e.g., code generation, writing).<br/><br/><strong>Code pattern:</strong><br/><code>approved = False; feedback = ""<br/>while not approved and iterations &lt; max_iter:<br/>&nbsp;&nbsp;out = await Runner.run(gen, original_input + feedback)<br/>&nbsp;&nbsp;eval_result = await Runner.run(eval, out.final_output)<br/>&nbsp;&nbsp;approved = eval_result.final_output.approved<br/>&nbsp;&nbsp;feedback = eval_result.final_output.feedback</code><br/><br/><code>Runner.run()</code> returns a <code>RunResult</code> object — always extract <code>.final_output</code> before passing output between agents. The evaluator agent requires an <code>output_type</code> (e.g. a Pydantic model with <code>approved: bool</code> and <code>feedback: str</code>) so its result can be parsed as structured data.'
  }
};
var orchCurrent = null;
function orchSelect(id) {
  var wrap = document.getElementById('orch-detail-wrap');
  var card = document.getElementById('orch-detail-card');
  document.querySelectorAll('.orch-node').forEach(function(n) { n.classList.toggle('active', n.dataset.orch === id); });
  if (orchCurrent === id) { wrap.classList.remove('open'); document.querySelectorAll('.orch-node').forEach(function(n) { n.classList.remove('active'); }); orchCurrent = null; return; }
  orchCurrent = id;
  var d = ORCH[id];
  card.style.borderTopColor = d.color;
  card.innerHTML = '<div style="font-size:16pt;font-weight:700;margin-bottom:12px;color:' + d.color + ';">' + d.title + '</div><div>' + d.html + '</div>';
  wrap.classList.add('open');
}
</script>

## B. Databricks MCP Tools with `databricks_openai.agents.McpServer`

For production agents, Databricks provides `McpServer` ([source](https://github.com/databricks/databricks-ai-bridge/blob/main/integrations/openai/src/databricks_openai/agents/mcp_server.py)) which connects agents to Unity Catalog functions, AI Search Indexes, and other Databricks services through the Model Context Protocol.

**Click on each layer** to see how the pieces connect.

<style>
.mcp-outer { max-width: 1100px; margin: 0 auto; font-family: sans-serif; font-size: 12pt; }
.mcp-diagram { display: flex; align-items: stretch; gap: 0; margin: 20px 0; }
.mcp-layer {
  flex: 1; padding: 14px 12px; text-align: center; cursor: pointer; user-select: none;
  transition: transform 0.12s, opacity 0.2s; border-right: 2px solid #fff;
}
.mcp-layer:last-child { border-right: none; }
.mcp-layer:hover { transform: translateY(-2px); }
.mcp-layer.active { opacity: 1; transform: translateY(-2px); box-shadow: 0 4px 12px rgba(0,0,0,0.15); }
.mcp-layer-title { font-size: 12pt; font-weight: 700; color: #fff; }
.mcp-layer-sub { font-size: 10pt; color: rgba(255,255,255,0.8); margin-top: 4px; }
.mcp-arrow { display: flex; justify-content: space-around; margin: 4px 0; font-size: 14px; color: #999; }
.mcp-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease; }
.mcp-detail-wrap.open { max-height: 2000px; opacity: 1; margin-top: 14px; }
.mcp-detail-card { background: #F9F7F4; border-radius: 10px; padding: 20px; border-top: 6px solid #ccc; font-size: 12pt; line-height: 1.7; color: #333; }
.mcp-detail-card strong { color: #0b2026; }
.mcp-detail-card code { background: #e8e8e8; padding: 2px 6px; border-radius: 4px; font-size: 11pt; font-family: 'Menlo','Consolas',monospace; }
.mcp-code { background: #1e1e2e; color: #cdd6f4; border-radius: 8px; padding: 14px 18px; margin-top: 12px; font-family: 'Menlo','Consolas',monospace; font-size: 11pt; line-height: 1.6; overflow-x: auto; }
.mcp-code .kw { color: #cba6f7; } .mcp-code .fn { color: #89b4fa; } .mcp-code .st { color: #a6e3a1; } .mcp-code .cm { color: #6c7086; }
</style>

<div class="mcp-outer">

<div class="mcp-diagram">
  <div class="mcp-layer" style="background:#FF5F46;border-radius:8px 0 0 8px;" data-mcp="agent" onclick="mcpSelect('agent')">
    <div class="mcp-layer-title">Agent</div>
    <div class="mcp-layer-sub">OpenAI Agents SDK</div>
  </div>
  <div class="mcp-layer" style="background:#4299E0;" data-mcp="mcpserver" onclick="mcpSelect('mcpserver')">
    <div class="mcp-layer-title">McpServer</div>
    <div class="mcp-layer-sub">databricks_openai.agents</div>
  </div>
  <div class="mcp-layer" style="background:#1B5162;" data-mcp="protocol" onclick="mcpSelect('protocol')">
    <div class="mcp-layer-title">MCP Protocol</div>
    <div class="mcp-layer-sub">Streamable HTTP</div>
  </div>
  <div class="mcp-layer" style="background:#00A972;border-radius:0 8px 8px 0;" data-mcp="uc" onclick="mcpSelect('uc')">
    <div class="mcp-layer-title">Unity Catalog</div>
    <div class="mcp-layer-sub">Functions, AI Search</div>
  </div>
</div>

<div class="mcp-arrow">
  <span>&#x2194; mcp_servers=[...]</span>
  <span>&#x2194; tools/list, tools/call</span>
  <span>&#x2194; /api/2.0/mcp/functions/...</span>
</div>

<div class="mcp-detail-wrap" id="mcp-detail-wrap">
  <div class="mcp-detail-card" id="mcp-detail-card"></div>
</div>

</div>

<script>
var MCP = {
  agent: {
    color: '#FF5F46',
    title: 'Agent Layer',
    html: '<strong>What it does:</strong> The agent receives <code>mcp_servers=[mcp_server]</code> and treats each MCP server as a source of tools. At runtime, the agent calls <code>tools/list</code> to discover available tools, then calls <code>tools/call</code> when the LLM decides to invoke one.<br/><br/><strong>Key point:</strong> The agent does not need to know what tools exist in advance. MCP provides runtime discovery.<br/><br/><div class="mcp-code"><span class="cm"># Agent with MCP tools</span><br/><span class="kw">from</span> agents <span class="kw">import</span> Agent, Runner<br/><span class="kw">from</span> databricks_openai.agents <span class="kw">import</span> McpServer<br/><br/><span class="kw">async with</span> <span class="fn">McpServer</span>(url=mcp_url) <span class="kw">as</span> server:<br/>&nbsp;&nbsp;&nbsp;&nbsp;agent = <span class="fn">Agent</span>(<br/>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;name=<span class="st">"my-agent"</span>,<br/>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;mcp_servers=[server],<br/>&nbsp;&nbsp;&nbsp;&nbsp;)<br/>&nbsp;&nbsp;&nbsp;&nbsp;result = <span class="kw">await</span> <span class="fn">Runner.run</span>(agent, messages)</div>'
  },
  mcpserver: {
    color: '#4299E0',
    title: 'McpServer Class',
    html: '<strong>What it does:</strong> <code>McpServer</code> extends the OpenAI Agents SDK <code>MCPServerStreamableHttp</code>. It adds Databricks authentication (<code>DatabricksOAuthClientProvider</code>), automatic URL construction, and MLflow tracing on every <code>call_tool</code>.<br/><br/><strong>Three factory methods:</strong><ul><li><code>McpServer(url=...)</code> -- direct URL to any MCP endpoint</li><li><code>McpServer.from_uc_function(catalog, schema, function_name)</code> -- builds URL: <code>/api/2.0/mcp/functions/{catalog}/{schema}/{function}</code></li><li><code>McpServer.from_vector_search(catalog, schema, index_name)</code> -- builds URL: <code>/api/2.0/mcp/vector-search/{catalog}/{schema}/{index}</code></li></ul><strong>Built-in tracing:</strong> Every tool call is wrapped with <code>@mlflow.trace(span_type=SpanType.TOOL)</code>, so MLflow automatically captures tool invocations as spans.<br/><br/><div class="mcp-code"><span class="cm"># Factory method for UC functions</span><br/>server = <span class="fn">McpServer.from_uc_function</span>(<br/>&nbsp;&nbsp;&nbsp;&nbsp;catalog=<span class="st">"main"</span>,<br/>&nbsp;&nbsp;&nbsp;&nbsp;schema=<span class="st">"tools"</span>,<br/>&nbsp;&nbsp;&nbsp;&nbsp;function_name=<span class="st">"get_top_products"</span>,<br/>&nbsp;&nbsp;&nbsp;&nbsp;timeout=30.0,<br/>)</div>'
  },
  protocol: {
    color: '#1B5162',
    title: 'MCP Protocol Layer',
    html: '<strong>What it does:</strong> The protocol layer handles the actual communication. <code>McpServer</code> uses Streamable HTTP transport (via <code>streamablehttp_client</code> from the <code>mcp</code> library) to connect to the server.<br/><br/><strong>Connection lifecycle:</strong><ol><li><code>async with McpServer(...) as server:</code> opens the HTTP connection</li><li>Agent calls <code>tools/list</code> to discover available tools</li><li>Agent calls <code>tools/call(tool_name, arguments)</code> to invoke a tool</li><li>Context manager exit closes the connection</li></ol><strong>Authentication:</strong> <code>McpServer</code> injects a <code>DatabricksOAuthClientProvider</code> into the HTTP transport. This handles token refresh automatically using the <code>WorkspaceClient</code> credentials. No manual token management needed.'
  },
  uc: {
    color: '#00A972',
    title: 'Unity Catalog Backend',
    html: '<strong>What it does:</strong> Databricks hosts Managed MCP servers that expose Unity Catalog resources as tools. The MCP server translates <code>tools/list</code> and <code>tools/call</code> into UC operations.<br/><br/><strong>Supported backends:</strong><table style="width:100%;border-collapse:collapse;margin-top:10px;font-size:12pt;"><tr style="border-bottom:1px solid #ddd;"><th style="text-align:left;padding:6px;">Type</th><th style="text-align:left;padding:6px;">URL Pattern</th><th style="text-align:left;padding:6px;">What It Exposes</th></tr><tr style="border-bottom:1px solid #ddd;"><td style="padding:6px;">UC Functions</td><td style="padding:6px;"><code>/api/2.0/mcp/functions/{cat}/{schema}</code></td><td style="padding:6px;">SQL and Python UC functions as callable tools</td></tr><tr style="border-bottom:1px solid #ddd;"><td style="padding:6px;">AI Search</td><td style="padding:6px;"><code>/api/2.0/mcp/vector-search/{cat}/{schema}</code></td><td style="padding:6px;">AI Search indexes as retrieval tools</td></tr><tr><td style="padding:6px;">Genie</td><td style="padding:6px;"><code>/api/2.0/mcp/genie/{space_id}</code></td><td style="padding:6px;">Genie spaces as natural language query tools</td></tr></table><br/><strong>Governance:</strong> UC permissions are always enforced. Agents can only access tools the authenticated user (or service principal) is authorized to use.'
  }
};
var mcpCurrent = null;
function mcpSelect(id) {
  var wrap = document.getElementById('mcp-detail-wrap');
  var card = document.getElementById('mcp-detail-card');
  document.querySelectorAll('.mcp-layer').forEach(function(n) { n.classList.toggle('active', n.dataset.mcp === id); });
  if (mcpCurrent === id) { wrap.classList.remove('open'); document.querySelectorAll('.mcp-layer').forEach(function(n) { n.classList.remove('active'); }); mcpCurrent = null; return; }
  mcpCurrent = id;
  var d = MCP[id];
  card.style.borderTopColor = d.color;
  card.innerHTML = '<div style="font-size:14pt;font-weight:700;margin-bottom:12px;color:' + d.color + ';">' + d.title + '</div><div>' + d.html + '</div>';
  wrap.classList.add('open');
}
</script>

## C. The @function_tool Decorator

For quick prototyping or agent-specific logic, the OpenAI Agents SDK provides the `@function_tool` decorator. This turns any Python function into a tool the agent can call:

```python
from agents import function_tool

@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"The weather in {city} is sunny, 72F."
```

The decorator uses the function's name, type hints, and docstring to generate the tool schema that the LLM sees. Clear docstrings and typed parameters improve tool selection accuracy.

## D. Multi-Agent Concepts

A single agent with tools is powerful, but many real-world workflows require **multiple specialized agents** working together. The OpenAI Agents SDK supports this natively through the `handoffs` parameter.

**Click on each concept** to learn how multi-agent systems work.

<style>
.ma-outer { max-width: 1100px; margin: 0 auto; font-family: sans-serif; font-size: 12pt; }
.ma-row { display: flex; flex-wrap: wrap; gap: 10px; margin-bottom: 14px; }
.ma-node {
  flex: 1; min-width: 160px; background: #F9F7F4; border-top: 6px solid transparent;
  border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent;
  border-radius: 8px; padding: 14px 12px; text-align: center; cursor: pointer; user-select: none;
  transition: transform 0.12s, background 0.15s;
}
.ma-node:hover { transform: translateY(-2px); }
.ma-node.active { background: #fff; border-left-color: var(--mc); border-right-color: var(--mc); border-bottom-color: var(--mc); }
.ma-node-title { display: block; font-size: 12pt; font-weight: 700; color: #0b2026; line-height: 1.3; pointer-events: none; }
.ma-node-sub { display: block; font-size: 12pt; color: #666; margin-top: 4px; pointer-events: none; }
.ma-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease; }
.ma-detail-wrap.open { max-height: 1600px; opacity: 1; margin-top: 12px; }
.ma-detail-card { background: #F9F7F4; border-radius: 10px; padding: 20px; border-top: 6px solid #ccc; font-size: 12pt; line-height: 1.7; color: #333; }
.ma-detail-card strong { color: #0b2026; }
.ma-detail-card code { background: #e8e8e8; padding: 2px 6px; border-radius: 4px; font-size: 12pt; }
.ma-flow { display: flex; align-items: center; justify-content: center; gap: 0; margin: 14px 0; flex-wrap: wrap; }
.ma-flow-box { background: #1e1e2e; color: #fff; border-radius: 6px; padding: 8px 14px; font-size: 12pt; font-weight: 600; text-align: center; min-width: 90px; }
.ma-flow-arrow { color: #999; font-size: 20px; padding: 0 6px; }
</style>

<div class="ma-outer">
<div class="ma-row">
  <div class="ma-node" data-ma="supervisor" onclick="maSelect('supervisor')" style="--mc:#FF5F46; border-top-color:#FF5F46;">
    <span class="ma-node-title">Supervisor Pattern</span>
    <span class="ma-node-sub">Route queries to specialists</span>
  </div>
  <div class="ma-node" data-ma="workers" onclick="maSelect('workers')" style="--mc:#4299E0; border-top-color:#4299E0;">
    <span class="ma-node-title">Worker Specialization</span>
    <span class="ma-node-sub">Each agent has focused tools</span>
  </div>
  <div class="ma-node" data-ma="mcp-multi" onclick="maSelect('mcp-multi')" style="--mc:#00A972; border-top-color:#00A972;">
    <span class="ma-node-title">MCP-Backed Workers</span>
    <span class="ma-node-sub">UC tools per agent via MCP</span>
  </div>
  <div class="ma-node" data-ma="tracing-multi" onclick="maSelect('tracing-multi')" style="--mc:#1B5162; border-top-color:#1B5162;">
    <span class="ma-node-title">Distributed Tracing</span>
    <span class="ma-node-sub">Unified trace across agents</span>
  </div>
</div>

<div class="ma-detail-wrap" id="ma-detail-wrap">
  <div class="ma-detail-card" id="ma-detail-card"></div>
</div>
</div>

<script>
var MA = {
  supervisor: {
    color: '#FF5F46',
    title: 'Supervisor Pattern',
    html: 'A supervisor agent receives all user queries and decides which worker to delegate to. The supervisor does not call tools directly. Instead, it uses <code>handoffs</code> to transfer the conversation to a specialist.<br/><br/><div class="ma-flow"><div class="ma-flow-box">User</div><div class="ma-flow-arrow">&#x2192;</div><div class="ma-flow-box" style="background:#FF5F46;">Supervisor</div><div class="ma-flow-arrow">&#x2192;</div><div style="display:flex;flex-direction:column;gap:4px;"><div class="ma-flow-box" style="background:#4299E0;">Data Analyst</div><div class="ma-flow-box" style="background:#00A972;">Business Analyst</div></div><div class="ma-flow-arrow">&#x2192;</div><div class="ma-flow-box">User</div></div><strong>SDK pattern:</strong> <code>Agent(name="Supervisor", handoffs=[data_analyst, business_analyst])</code><br/><br/>The supervisor reads each worker\'s name and instructions to decide which one is best suited for the request.'
  },
  workers: {
    color: '#4299E0',
    title: 'Worker Specialization',
    html: 'Each worker agent has a focused role with its own instructions and tools. This keeps prompts concise and tool sets relevant.<br/><br/><strong>Example of specialized agents:</strong><ul><li><strong>Data Analyst</strong> - uses <code>classify_price_tier</code> to categorize Airbnb prices</li><li><strong>Business Analyst</strong> - uses <code>neighborhood_summary</code> for market insights</li></ul><strong>Why specialize?</strong> A single agent with many tools may struggle to select the right one. Specialized agents with 1-2 tools each make more accurate tool selections and produce more focused responses.'
  },
  'mcp-multi': {
    color: '#00A972',
    title: 'MCP-Backed Workers',
    html: 'Each worker agent connects to its own <code>McpServer</code> instance, scoped to the specific UC function it needs. This means each agent only sees the tools relevant to its role.<br/><br/><strong>Pattern:</strong><br/><code>da_mcp = McpServer.from_uc_function(catalog, schema, "classify_price_tier")</code><br/><code>ba_mcp = McpServer.from_uc_function(catalog, schema, "neighborhood_summary")</code><br/><br/><code>data_analyst = Agent(mcp_servers=[da_mcp])</code><br/><code>biz_analyst = Agent(mcp_servers=[ba_mcp])</code><br/><br/><strong>Key point:</strong> The <code>async with</code> context manager must wrap the entire multi-agent flow so all MCP connections stay open during execution.'
  },
  'tracing-multi': {
    color: '#1B5162',
    title: 'Distributed Tracing',
    html: '<strong>Challenge:</strong> With <code>mlflow.openai.autolog()</code> alone, each agent invocation produces a separate trace. The supervisor call and each worker call appear as independent traces, making it hard to see the full orchestration.<br/><br/><strong>Solution:</strong> Wrap the entire multi-agent flow in a parent <code>@mlflow.trace</code> span. All child spans (routing, worker LLM calls, MCP tool calls) nest under this root, producing a single trace tree.<br/><br/><code>@mlflow.trace(span_type="AGENT", name="supervisor_orchestration")<br/>async def run_supervisor(question):<br/>&nbsp;&nbsp;# All agent calls inside become child spans<br/>&nbsp;&nbsp;result = await Runner.run(supervisor, question)<br/>&nbsp;&nbsp;return result.final_output</code>'
  }
};
var maCurrent = null;
function maSelect(id) {
  var wrap = document.getElementById('ma-detail-wrap');
  var card = document.getElementById('ma-detail-card');
  document.querySelectorAll('.ma-node').forEach(function(n) { n.classList.toggle('active', n.dataset.ma === id); });
  if (maCurrent === id) { wrap.classList.remove('open'); document.querySelectorAll('.ma-node').forEach(function(n) { n.classList.remove('active'); }); maCurrent = null; return; }
  maCurrent = id;
  var d = MA[id];
  card.style.borderTopColor = d.color;
  card.innerHTML = '<div style="font-size:16pt;font-weight:700;margin-bottom:12px;color:' + d.color + ';">' + d.title + '</div><div>' + d.html + '</div>';
  wrap.classList.add('open');
}
</script>

## E. Other Supported Frameworks

While this course focuses on the OpenAI Agents SDK, Databricks supports multiple agent authoring frameworks. The key requirement is wrapping your agent with the MLflow `ResponsesAgent` interface for deployment. Some other popular frameworks are shown below. 

**Click on each framework** to learn about its strengths and integration with Databricks.

<style>
.fw-outer { max-width: 1100px; margin: 0 auto; font-family: sans-serif; font-size: 12pt; }
.fw-row { display: flex; flex-wrap: wrap; gap: 10px; margin-bottom: 14px; }
.fw-node {
  flex: 1; min-width: 140px; background: #F9F7F4; border-top: 6px solid transparent;
  border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent;
  border-radius: 8px; padding: 14px 12px; text-align: center; cursor: pointer; user-select: none;
  transition: transform 0.12s, background 0.15s;
}
.fw-node:hover { transform: translateY(-2px); }
.fw-node.active { background: #fff; border-left-color: var(--fc); border-right-color: var(--fc); border-bottom-color: var(--fc); }
.fw-node-title { display: block; font-size: 12pt; font-weight: 700; color: #0b2026; line-height: 1.3; pointer-events: none; }
.fw-node-sub { display: block; font-size: 12pt; color: #666; margin-top: 4px; pointer-events: none; }
.fw-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease; }
.fw-detail-wrap.open { max-height: 1600px; opacity: 1; margin-top: 12px; }
.fw-detail-card { background: #F9F7F4; border-radius: 10px; padding: 20px; border-top: 6px solid #ccc; font-size: 12pt; line-height: 1.7; color: #333; }
.fw-detail-card strong { color: #0b2026; }
.fw-detail-card code { background: #e8e8e8; padding: 2px 6px; border-radius: 4px; font-size: 12pt; }
.fw-code { background: #1e1e2e; color: #cdd6f4; border-radius: 8px; padding: 14px 18px; margin-top: 12px; font-family: 'Menlo','Consolas',monospace; font-size: 11pt; line-height: 1.6; overflow-x: auto; }
.fw-code .kw { color: #cba6f7; } .fw-code .fn { color: #89b4fa; } .fw-code .st { color: #a6e3a1; } .fw-code .cm { color: #6c7086; }
</style>

<div class="fw-outer">
<div class="fw-row">
  <div class="fw-node" data-fw="langchain" onclick="fwSelect('langchain')" style="--fc:#FF5F46; border-top-color:#FF5F46;">
    <span class="fw-node-title">LangChain</span>
    <span class="fw-node-sub">Composable chains and tools</span>
  </div>
  <div class="fw-node" data-fw="langgraph" onclick="fwSelect('langgraph')" style="--fc:#4299E0; border-top-color:#4299E0;">
    <span class="fw-node-title">LangGraph</span>
    <span class="fw-node-sub">Stateful graph workflows</span>
  </div>
  <div class="fw-node" data-fw="dspy" onclick="fwSelect('dspy')" style="--fc:#00A972; border-top-color:#00A972;">
    <span class="fw-node-title">DSPy</span>
    <span class="fw-node-sub">Prompt optimization</span>
  </div>
</div>

<div class="fw-detail-wrap" id="fw-detail-wrap">
  <div class="fw-detail-card" id="fw-detail-card"></div>
</div>
</div>

<script>
var FW = {
  langchain: {
    color: '#FF5F46',
    title: 'LangChain',
    html: '<strong>What it is:</strong> A composable framework for building LLM applications using chains, agents, and tools.<br/><br/><strong>Install:</strong><br/><code>%pip install unitycatalog-langchain[databricks]</code><br/><code>%pip install databricks-langchain</code><br/><br/><strong>Databricks integration:</strong><ul><li><code>databricks-langchain</code> provides <strong>ChatDatabricks</strong> and <strong>UCFunctionToolkit</strong> to expose Unity Catalog functions as LangChain tools.</li><li>Use <strong>AgentExecutor</strong> + <strong>create_tool_calling_agent</strong> to build the agent loop.</li><li>Use <strong><code>mlflow.langchain.autolog()</code></strong> to enable automatic tracing for the LangChain app.</li></ul><em>Note: Databricks now recommends MCP-based tools for most new agents. UCFunctionToolkit remains supported for UC function tools.</em><br/><br/><div class="fw-code"><span class="kw">import</span> mlflow<br/><span class="kw">from</span> databricks_langchain <span class="kw">import</span> ChatDatabricks, UCFunctionToolkit<br/><span class="kw">from</span> langchain.agents <span class="kw">import</span> create_tool_calling_agent, AgentExecutor<br/><span class="kw">from</span> langchain.prompts <span class="kw">import</span> ChatPromptTemplate<br/><br/>LLM_ENDPOINT_NAME = <span class="st">"databricks-gpt-5-2"</span><br/><br/>llm = <span class="fn">ChatDatabricks</span>(endpoint=LLM_ENDPOINT_NAME)<br/>toolkit = <span class="fn">UCFunctionToolkit</span>(function_names=[<span class="st">"catalog.schema.my_func"</span>])<br/>tools = toolkit.tools<br/><br/>prompt = <span class="fn">ChatPromptTemplate.from_messages</span>([<br/>&nbsp;&nbsp;(<span class="st">"system"</span>, <span class="st">"You are a helpful assistant."</span>),<br/>&nbsp;&nbsp;(<span class="st">"placeholder"</span>, <span class="st">"{chat_history}"</span>),<br/>&nbsp;&nbsp;(<span class="st">"human"</span>, <span class="st">"{input}"</span>),<br/>&nbsp;&nbsp;(<span class="st">"placeholder"</span>, <span class="st">"{agent_scratchpad}"</span>),<br/>])<br/><br/>mlflow.langchain.<span class="fn">autolog</span>()<br/><br/>agent = <span class="fn">create_tool_calling_agent</span>(llm, tools, prompt)<br/>agent_executor = <span class="fn">AgentExecutor</span>(agent=agent, tools=tools)<br/>agent_executor.<span class="fn">invoke</span>({<span class="st">"input"</span>: <span class="st">"What is the average price in Mission?"</span>})</div>'
  },
  langgraph: {
    color: '#4299E0',
    title: 'LangGraph',
    html: '<strong>What it is:</strong> An open-source library for building stateful, multi-actor applications with LLMs, used to create agent and multi-agent workflows. Agents are defined as nodes in a directed graph with explicit state management and conditional edges.<br/><br/><strong>Databricks integration:</strong><ul><li>Use <strong>ChatDatabricks</strong> from <code>databricks-langchain</code> with <strong>create_react_agent</strong> from <code>langgraph.prebuilt</code></li><li>Stateful, multi-actor agent graphs suitable for long-running workflows (LangGraph provides persistence/checkpointing features)</li><li>Tools can come from managed MCP servers via <code>DatabricksMCPServer</code> + <code>DatabricksMultiServerMCPClient</code>, then passed into <code>create_react_agent</code></li><li><code>mlflow.langchain.autolog()</code> traces LangGraph execution, capturing graph steps as spans</li></ul><div class="fw-code"><span class="kw">import</span> mlflow<br/><span class="kw">from</span> langchain_core.tools <span class="kw">import</span> tool<br/><span class="kw">from</span> langgraph.prebuilt <span class="kw">import</span> create_react_agent<br/><span class="kw">from</span> databricks_langchain <span class="kw">import</span> ChatDatabricks<br/><br/><span class="cm"># Define a tool</span><br/>@tool<br/><span class="kw">def</span> <span class="fn">get_current_time</span>() -> str:<br/>&nbsp;&nbsp;<span class="st">"""Get the current date and time."""</span><br/>&nbsp;&nbsp;<span class="kw">from</span> datetime <span class="kw">import</span> datetime<br/>&nbsp;&nbsp;<span class="kw">return</span> datetime.<span class="fn">now</span>().<span class="fn">isoformat</span>()<br/><br/>mlflow.langchain.<span class="fn">autolog</span>()<br/><br/>llm = <span class="fn">ChatDatabricks</span>(endpoint=<span class="st">"databricks-claude-sonnet-4-5"</span>)<br/>agent = <span class="fn">create_react_agent</span>(llm, tools=[get_current_time])</div>'
  },
  dspy: {
    color: '#00A972',
    title: 'DSPy',
    html: '<strong>What it is:</strong> A framework for programmatically defining and optimizing generative AI agents. You specify input/output signatures and DSPy\'s compiler automatically optimizes prompts and, when applicable, model weights for your task and data.<br/><br/><strong>Databricks integration:</strong><ul><li><code>mlflow.dspy.autolog()</code> captures DSPy program, evaluation, and compilation traces into MLflow on Databricks</li><li>Databricks provides DSPy example notebooks for text classification, RAG over AI Search, multi-agent systems with Genie + DSPy, and migrating LangChain to DSPy</li><li>For production agents, use the <code>databricks-dspy</code> package alongside MLflow <code>ResponsesAgent</code></li></ul><div class="fw-code"><span class="kw">import</span> mlflow<br/><span class="kw">import</span> dspy<br/><br/>mlflow.dspy.<span class="fn">autolog</span>()<br/>mlflow.<span class="fn">set_tracking_uri</span>(<span class="st">"databricks"</span>)<br/>mlflow.<span class="fn">set_experiment</span>(<span class="st">"/Shared/dspy-tracing-demo"</span>)<br/><br/>lm = dspy.<span class="fn">LM</span>(<span class="st">"databricks/databricks-gpt-5-5"</span>)<br/>dspy.<span class="fn">configure</span>(lm=lm)<br/><br/><span class="cm"># Define a signature, not a prompt</span><br/><span class="kw">class</span> <span class="fn">QA</span>(dspy.Signature):<br/>&nbsp;&nbsp;question: str = dspy.<span class="fn">InputField</span>()<br/>&nbsp;&nbsp;answer: str = dspy.<span class="fn">OutputField</span>()<br/><br/>predictor = dspy.<span class="fn">Predict</span>(QA)</div>'
  }
};
var fwCurrent = null;
function fwSelect(id) {
  var wrap = document.getElementById('fw-detail-wrap');
  var card = document.getElementById('fw-detail-card');
  document.querySelectorAll('.fw-node').forEach(function(n) { n.classList.toggle('active', n.dataset.fw === id); });
  if (fwCurrent === id) { wrap.classList.remove('open'); document.querySelectorAll('.fw-node').forEach(function(n) { n.classList.remove('active'); }); fwCurrent = null; return; }
  fwCurrent = id;
  var d = FW[id];
  card.style.borderTopColor = d.color;
  card.innerHTML = '<div style="font-size:16pt;font-weight:700;margin-bottom:12px;color:' + d.color + ';">' + d.title + '</div><div>' + d.html + '</div>';
  wrap.classList.add('open');
}
</script>

## F. MLflow Integration

<div style="width: 100%; font-family: sans-serif;">

<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">

  <img src="./Includes/images/lecture 1/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Want to know more about how MLflow integrates with the OpenAI Agent SDK on Databricks? Ask Genie Code. Click on the genie icon <img src="./Includes/images/lecture 1/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">What does set_trace_processors([]) from the OpenAI Agents SDK do? Why is it needed with MLflow?</span>
    <button onclick="
      var text = document.getElementById('genie-query').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>

### F1. Traditional ML vs Agents

For robust operations with GenAI applications and agents, you often need deeper insight into **server-side behavior**, including **latency/cost metrics** and **API logging**, beyond what is typically required for traditional ML batch workloads.

| | Traditional ML | Agents |
|---|---|---|
| **Shape** | Single call | Multi-step, often looped |
| **Visible by default** | Input, output | Input, final output |
| **Unit of observation** | The call | A **span** per step |
| **Needed to debug** | Features, latency | Per-step I/O, tokens, tool args |
| **MLflow captures** | Request / response | Full **trace** (tree of spans) |

#### Agents need more insight
- Agents perform multiple intermediate steps (for example: retrieval, tool use, LLM calls), and you need to see each step, its inputs/outputs, and per-step latency/token usage to debug and improve quality.
- **MLflow Tracing** captures these as **traces** and **spans** automatically for supported libraries (such as the OpenAI SDK, LangChain, and other GenAI frameworks) and provides a UI and APIs to analyze them across development and production.
- In a **tracing system** such as OpenTelemetry and MLflow, a single operation is represented as a **span**. It records when the operation started and ended, along with the metadata, inputs, and outputs for that unit of work.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <div>
      <strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
      <p style="margin: 8px 0 0 0; color: #333;">
        MLflow spans are designed to be compatible with the
        <a href="https://opentelemetry.io/docs/concepts/signals/traces/" style="color: #1976d2; text-decoration: underline;">OpenTelemetry trace specification</a>, so additional information (such as token counts) is stored as key-value <em>attributes</em> on the span, rather than as new top-level fields.
      </p>
    </div>
  </div>
</div>


### F2. Hierarchical Span Structure

MLflow organizes trace data using a hierarchical span structure that mirrors agent execution, starting with a single root span that represents the overall request or workflow, with nested child spans for each sub-step.

- **Parent spans**: High-level operations like "process user request"
- **Child spans**: Detailed steps like "call retrieval tool" or "generate response"
- **Span relationships**: Clear parent-child relationships that show execution flow, which should mimic your application's execution plan.
- **Span types**: Categorization of spans (`TOOL`, `CHAT_MODEL`, `RETRIEVER`) for better organization

![tracing-example.png](./Includes/images/tracing-example.png "tracing-example.png")
<p>
</p>


## G. Conclusion

In this lecture, you learned the building blocks for agent development on Databricks:

- **Orchestration patterns**: handoffs, agents-as-tools, sequential, parallel, and feedback loops
- **MCP integration**: `McpServer` connects agents to governed UC functions at runtime
- **`@function_tool`**: rapid prototyping of agent tools without UC registration
- **Multi-agent concepts**: supervisor routing, worker specialization, and MCP-scoped tools
- **Other frameworks**: LangChain, LangGraph, and DSPy as alternative authoring options
- **MLflow tracing**: hierarchical spans that capture every step of agent execution


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>